# LeetCode 3130 — Find All Possible Stable Binary Arrays II bằng Python

- Link: https://leetcode.com/problems/find-all-possible-stable-binary-arrays-ii/
- Độ khó: **Hard**
- Chủ đề: **DP đếm theo (số 0, số 1, chữ số cuối)** với ràng buộc "không có khối nào dài hơn `limit`" — loại các chuỗi vi phạm bằng **phép trừ** (inclusion–exclusion một bước) ⇒ `O(zero · one)`; cách khác: **tổ hợp** chia mỗi loại thành các khối

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfStableArrays(self, zero: int, one: int, limit: int) -> int:
```

## 1. Mục tiêu học

1. "Mọi mảng con dài hơn `limit` chứa cả 0 và 1" ⇔ **không có khối** (đoạn liên tiếp cùng chữ số) dài hơn `limit`.
2. DP `f0[i][j]` / `f1[i][j]` = số mảng dùng `i` số 0, `j` số 1, **kết thúc bằng** 0 / 1, mọi khối `≤ limit`.
3. Chuyển trực tiếp: thêm một khối `t ∈ [1, limit]` chữ số cuối ⇒ `O(zero · one · limit)`.
4. Tối ưu bằng phép trừ: `f0[i][j] = f0[i−1][j] + f1[i−1][j] − f1[i−1−limit][j]` (bỏ các mảng vừa tạo khối 0 dài `limit + 1`) ⇒ `O(zero · one)`.
5. Cách tổ hợp: số cách chia `n` thành `k` phần trong `[1, limit]` bằng inclusion–exclusion; ghép số khối 0 và khối 1 lệch nhau tối đa 1.

## 2. Đề bài & ví dụ

Cho `zero`, `one`, `limit`. Mảng nhị phân **ổn định** nếu có đúng `zero` số 0, đúng `one` số 1, và mọi mảng con độ dài `> limit` chứa cả 0 lẫn 1. Đếm số mảng ổn định mod `10^9 + 7`.

**Ví dụ 1:** `(1, 1, 2)` → `2`. **Ví dụ 2:** `(1, 2, 1)` → `1`. **Ví dụ 3:** `(3, 3, 2)` → `14`.

**Ràng buộc:** `1 ≤ zero, one, limit ≤ 1000`.

In [1]:
from functools import lru_cache
from math import comb
import itertools, random, time

MOD = 10 ** 9 + 7
examples = [(1, 1, 2, 2), (1, 2, 1, 1), (3, 3, 2, 14)]
for z, o, L, exp in examples:
    print(f'zero = {z}, one = {o}, limit = {L} -> kỳ vọng {exp}')

zero = 1, one = 1, limit = 2 -> kỳ vọng 2
zero = 1, one = 2, limit = 1 -> kỳ vọng 1
zero = 3, one = 3, limit = 2 -> kỳ vọng 14


## 3. Kiến thức nền

### 3.1. Ràng buộc mảng con ⇔ ràng buộc khối

Một mảng con dài `> limit` **thiếu** một chữ số ⇔ nó nằm trong một khối dài `> limit`. Vậy điều kiện tương đương: mọi khối có độ dài `≤ limit`.

In [2]:
def stable_by_definition(arr, limit):
    n = len(arr)
    return all(len(set(arr[i:j])) == 2 for i in range(n) for j in range(i + limit + 1, n + 1))

def stable_by_runs(arr, limit):
    return all(len(list(g)) <= limit for _, g in itertools.groupby(arr))

ok = all(stable_by_definition(a, L) == stable_by_runs(a, L)
         for n in range(1, 9) for a in itertools.product((0, 1), repeat=n) for L in range(1, 5))
print('Định nghĩa theo mảng con ⇔ mọi khối ≤ limit:', ok)

Định nghĩa theo mảng con ⇔ mọi khối ≤ limit: True


### 3.2. Phép trừ để bỏ khối quá dài

`g0[i][j]` (chưa có ràng buộc riêng cho khối cuối) = số mảng hợp lệ dùng `i−1` số 0, `j` số 1 rồi thêm một số 0. Trong đó có những mảng mà khối 0 cuối giờ dài **đúng** `limit + 1` — chúng có dạng (mảng hợp lệ kết thúc bằng 1 với `i − 1 − limit` số 0) + `limit + 1` số 0. Trừ đi số đó: `f1[i−1−limit][j]`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xây mảng từ trái sang phải; trạng thái cần biết: còn bao nhiêu 0/1 đã dùng, chữ số cuối, và **khối cuối dài bao nhiêu**. Thay vì nhớ độ dài khối, ta "thêm cả khối một lúc" (cách chuyển `O(limit)`), rồi nhận ra tổng các khối liên tiếp là một **tổng trượt** ⇒ thay bằng phép trừ.

Ví dụ 2 (`zero = 1, one = 2, limit = 1`): khối 1 không được dài hơn 1 ⇒ chỉ có `1, 0, 1`.

## 5. Cách 1 — Brute Force: liệt kê vị trí các số 0

In [3]:
def brute(zero, one, limit):
    n = zero + one; cnt = 0
    for pos in itertools.combinations(range(n), zero):
        arr = [1] * n
        for p in pos:
            arr[p] = 0
        cnt += stable_by_runs(arr, limit)
    return cnt % MOD

for z, o, L, exp in examples:
    print('brute:', brute(z, o, L), '| kỳ vọng', exp)

brute: 2 | kỳ vọng 2
brute: 1 | kỳ vọng 1
brute: 14 | kỳ vọng 14


**Độ phức tạp:** `O(C(zero + one, zero) · n)` — chỉ cho `zero + one ≤ 16`.

## 6. Ý tưởng thuật toán / bước đột phá

```
f0[i][0] = 1 nếu 1 ≤ i ≤ limit;  f1[0][j] = 1 nếu 1 ≤ j ≤ limit
f0[i][j] = f0[i-1][j] + f1[i-1][j] − (f1[i-1-limit][j] nếu i-1-limit ≥ 0)
f1[i][j] = f0[i][j-1] + f1[i][j-1] − (f0[i][j-1-limit] nếu j-1-limit ≥ 0)
ans = f0[zero][one] + f1[zero][one]
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP thêm cả khối `O(zero · one · limit)` (bản I)

In [4]:
def dp_blocks(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(zero + 1):
        for j in range(one + 1):
            if i == 0 and j == 0:
                continue
            # kết thúc bằng một khối 0 dài t
            s = 0
            for t in range(1, min(limit, i) + 1):
                if i - t == 0 and j == 0:
                    s += 1
                else:
                    s += f1[i - t][j]
            f0[i][j] = s % MOD
            s = 0
            for t in range(1, min(limit, j) + 1):
                if j - t == 0 and i == 0:
                    s += 1
                else:
                    s += f0[i][j - t]
            f1[i][j] = s % MOD
    return (f0[zero][one] + f1[zero][one]) % MOD

for z, o, L, exp in examples:
    print('DP thêm khối:', dp_blocks(z, o, L), '| kỳ vọng', exp)

DP thêm khối: 2 | kỳ vọng 2
DP thêm khối: 1 | kỳ vọng 1
DP thêm khối: 14 | kỳ vọng 14


### 7.2. Cách 3 — DP với phép trừ `O(zero · one)` (lời giải chính)

In [5]:
def dp_subtract(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, min(zero, limit) + 1):
        f0[i][0] = 1
    for j in range(1, min(one, limit) + 1):
        f1[0][j] = 1
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            v = f0[i - 1][j] + f1[i - 1][j]
            if i - 1 - limit >= 0:
                v -= f1[i - 1 - limit][j]
            f0[i][j] = v % MOD
            v = f0[i][j - 1] + f1[i][j - 1]
            if j - 1 - limit >= 0:
                v -= f0[i][j - 1 - limit]
            f1[i][j] = v % MOD
    return (f0[zero][one] + f1[zero][one]) % MOD

for z, o, L, exp in examples:
    print('DP trừ:', dp_subtract(z, o, L), '| kỳ vọng', exp)

DP trừ: 2 | kỳ vọng 2
DP trừ: 1 | kỳ vọng 1
DP trừ: 14 | kỳ vọng 14


Kiểm tra phép trừ khi `i − 1 − limit = 0`: mảng "toàn 1 rồi thêm `limit + 1` số 0" có `f1[0][j] = 1` (nếu `j ≤ limit`) — đúng là một mảng cần loại. Còn mảng "chỉ toàn 0" (`j = 0`) không vào vòng lặp vì `j ≥ 1`.

### 7.3. Cách 4 — Tổ hợp: chia thành khối

Một mảng = các khối xen kẽ. Nếu có `a` khối 0 và `b` khối 1 thì `|a − b| ≤ 1`; số cách sắp: `a = b` ⇒ 2 (bắt đầu bằng 0 hoặc 1), `|a − b| = 1` ⇒ 1. Số cách chia `n` thành `k` phần mỗi phần trong `[1, L]`:

`P(n, k, L) = Σ_t (−1)^t C(k, t) C(n − tL − 1, k − 1)` (inclusion–exclusion trên các phần vượt `L`).

In [6]:
def parts(n, k, L):
    if k == 0:
        return 1 if n == 0 else 0
    s = 0
    for t in range(0, k + 1):
        m = n - t * L
        if m < k:
            break
        term = comb(k, t) * comb(m - 1, k - 1)
        s += -term if t & 1 else term
    return s

def combinatorial(zero, one, limit):
    P0 = [parts(zero, a, limit) for a in range(zero + 2)]
    P1 = [parts(one, b, limit) for b in range(one + 2)]
    ans = 0
    for a in range(1, zero + 1):
        if P0[a] == 0:
            continue
        for b in (a - 1, a, a + 1):
            if 1 <= b <= one:
                ans += P0[a] * P1[b] * (2 if a == b else 1)
    return ans % MOD

for z, o, L, exp in examples:
    print('tổ hợp:', combinatorial(z, o, L), '| kỳ vọng', exp)

tổ hợp: 2 | kỳ vọng 2
tổ hợp: 1 | kỳ vọng 1
tổ hợp: 14 | kỳ vọng 14


## 8. TRACE từng bước

In bảng `f0`, `f1` cho ví dụ 3 (`zero = one = 3`, `limit = 2`).

In [7]:
def tables(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, min(zero, limit) + 1): f0[i][0] = 1
    for j in range(1, min(one, limit) + 1): f1[0][j] = 1
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            f0[i][j] = f0[i - 1][j] + f1[i - 1][j] - (f1[i - 1 - limit][j] if i - 1 - limit >= 0 else 0)
            f1[i][j] = f0[i][j - 1] + f1[i][j - 1] - (f0[i][j - 1 - limit] if j - 1 - limit >= 0 else 0)
    return f0, f1

f0, f1 = tables(3, 3, 2)
print('f0 (hàng i = số 0, cột j = số 1):'); [print('  ', r) for r in f0]
print('f1:'); [print('  ', r) for r in f1]
print('đáp án f0[3][3] + f1[3][3] =', f0[3][3] + f1[3][3])

f0 (hàng i = số 0, cột j = số 1):
   [0, 0, 0, 0]
   [1, 1, 1, 0]
   [1, 2, 3, 2]
   [0, 2, 5, 7]
f1:
   [0, 1, 1, 0]
   [0, 1, 2, 2]
   [0, 1, 3, 5]
   [0, 0, 2, 7]
đáp án f0[3][3] + f1[3][3] = 14


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def numberOfStableArrays(self, zero: int, one: int, limit: int) -> int:
        MOD = 10 ** 9 + 7
        f0 = [[0] * (one + 1) for _ in range(zero + 1)]
        f1 = [[0] * (one + 1) for _ in range(zero + 1)]
        for i in range(1, min(zero, limit) + 1):
            f0[i][0] = 1
        for j in range(1, min(one, limit) + 1):
            f1[0][j] = 1
        for i in range(1, zero + 1):
            a0 = f0[i - 1]
            a1 = f1[i - 1]
            b1 = f1[i - 1 - limit] if i - 1 - limit >= 0 else None
            c0 = f0[i]
            c1 = f1[i]
            for j in range(1, one + 1):
                v = a0[j] + a1[j]
                if b1 is not None:
                    v -= b1[j]
                c0[j] = v % MOD
                v = c0[j - 1] + c1[j - 1]
                if j - 1 - limit >= 0:
                    v -= c0[j - 1 - limit]
                c1[j] = v % MOD
        return (f0[zero][one] + f1[zero][one]) % MOD

for z, o, L, exp in examples:
    got = Solution().numberOfStableArrays(z, o, L)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
14 OK


## 10. Vì sao thuật toán đúng

1. **Tương đương khối:** mục 3.1.
2. **Bất biến:** `f0[i][j]` = số mảng hợp lệ (mọi khối `≤ limit`) dùng `i` số 0, `j` số 1, kết thúc bằng 0. Bỏ số 0 cuối của một mảng như vậy được một mảng dùng `i−1` số 0: hoặc rỗng (chỉ khi `j = 0`, đã xử lý ở hàng cơ sở), hoặc hợp lệ kết thúc bằng 0 (khối 0 cuối `< limit`), hoặc hợp lệ kết thúc bằng 1. Ngược lại thêm 0 vào mọi mảng trong `f0[i−1][j] + f1[i−1][j]` cho mảng kết thúc bằng 0 — hợp lệ **trừ** khi khối 0 cuối thành `limit + 1`; các mảng hỏng đó ứng một-một với "mảng hợp lệ kết thúc bằng 1 dùng `i − 1 − limit` số 0" + `limit + 1` số 0 ⇒ trừ `f1[i−1−limit][j]`.
3. Đối xứng cho `f1`. Đáp án `f0 + f1` ở ô cuối.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(C(n, zero) · n)` | `O(n)` |
| 2. DP thêm khối | `O(zero · one · limit)` | `O(zero · one)` |
| 3. DP trừ | `O(zero · one)` | `O(zero · one)` |
| 4. Tổ hợp | `O(Σ_k k) ≈ O(zero² + one²)` phép `comb` | `O(zero + one)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'blocks': dp_blocks, 'subtract': dp_subtract, 'comb': combinatorial,
         'submit': lambda z, o, L: Solution().numberOfStableArrays(z, o, L)}
edge = [(1, 1, 1, 2), (2, 1, 1, 1), (3, 1, 1, 0), (2, 2, 1, 2), (4, 4, 4, 70), (5, 1, 3, 2), (3, 3, 1, 2)]
for z, o, L, exp in examples + edge:
    assert brute(z, o, L) == exp, (z, o, L, brute(z, o, L))
    for name, f in impls.items():
        assert f(z, o, L) == exp, (name, z, o, L)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `(3, 1, 1)` — ba số 0 chỉ được tách bởi một số 1 ⇒ luôn có khối 0 dài `≥ 2` ⇒ `0`. `(4, 4, 4)` — `limit ≥` mọi khối có thể ⇒ mọi `C(8, 4) = 70` mảng. `(5, 1, 3)`: số 1 phải chia 5 số 0 thành hai khối `≤ 3` ⇒ `(2, 3)` hoặc `(3, 2)`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3130)
cnt = 0
for _ in range(300):
    z = rng.randint(1, 8); o = rng.randint(1, 8); L = rng.randint(1, 5)
    exp = brute(z, o, L)
    for name, f in impls.items():
        assert f(z, o, L) == exp, (name, z, o, L)
    cnt += 1
for _ in range(30):
    z = rng.randint(1, 60); o = rng.randint(1, 60); L = rng.randint(1, 20)
    assert dp_blocks(z, o, L) == dp_subtract(z, o, L) == combinatorial(z, o, L)
print('Khớp brute trên', cnt, 'bộ; ba cách DP/tổ hợp khớp nhau với zero, one ≤ 60.')

Khớp brute trên 300 bộ; ba cách DP/tổ hợp khớp nhau với zero, one ≤ 60.


### 12.3. Ca lớn `zero = one = 1000`

In [11]:
for z, o, L in [(1000, 1000, 1), (1000, 1000, 7), (1000, 1000, 1000), (1000, 500, 3)]:
    t = time.time(); x = Solution().numberOfStableArrays(z, o, L); t1 = time.time() - t
    t = time.time(); y = combinatorial(z, o, L); t2 = time.time() - t
    assert x == y
    print(f'({z}, {o}, {L}): {x:10d} | nộp {t1:.2f}s | tổ hợp {t2:.2f}s')
assert Solution().numberOfStableArrays(1000, 1000, 1) == 2
assert Solution().numberOfStableArrays(1000, 1000, 1000) == comb(2000, 1000) % MOD
print('limit = 1 ⇒ chỉ 0101… và 1010…; limit = 1000 ⇒ C(2000, 1000): đúng.')

(1000, 1000, 1):          2 | nộp 0.14s | tổ hợp 5.73s


(1000, 1000, 7):    5161670 | nộp 0.26s | tổ hợp 0.91s
(1000, 1000, 1000):   72475738 | nộp 0.18s | tổ hợp 0.02s


(1000, 500, 3):  446272873 | nộp 0.10s | tổ hợp 1.12s


limit = 1 ⇒ chỉ 0101… và 1010…; limit = 1000 ⇒ C(2000, 1000): đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trừ ở chỉ số `i − limit` thay vì `i − 1 − limit`

In [12]:
def wrong_index(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, min(zero, limit) + 1): f0[i][0] = 1
    for j in range(1, min(one, limit) + 1): f1[0][j] = 1
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            f0[i][j] = (f0[i - 1][j] + f1[i - 1][j] - (f1[i - limit][j] if i - limit >= 0 else 0)) % MOD     # SAI
            f1[i][j] = (f0[i][j - 1] + f1[i][j - 1] - (f0[i][j - limit] if j - limit >= 0 else 0)) % MOD     # SAI
    return (f0[zero][one] + f1[zero][one]) % MOD

print('sai :', wrong_index(3, 3, 2), '| đúng:', Solution().numberOfStableArrays(3, 3, 2))

sai : 2 | đúng: 14


**Vì sao sai:** mảng bị loại là mảng có khối dài `limit + 1` — tiền tố trước khối đó dùng `i − (limit + 1)` số 0. Lệch một chỉ số loại nhầm các mảng có khối dài đúng `limit` (hợp lệ).

### Lỗi 2 — Hàng cơ sở không giới hạn bởi `limit`

In [13]:
def wrong_base(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, zero + 1): f0[i][0] = 1                                                           # SAI
    for j in range(1, one + 1): f1[0][j] = 1                                                            # SAI
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            f0[i][j] = (f0[i - 1][j] + f1[i - 1][j] - (f1[i - 1 - limit][j] if i - 1 - limit >= 0 else 0)) % MOD
            f1[i][j] = (f0[i][j - 1] + f1[i][j - 1] - (f0[i][j - 1 - limit] if j - 1 - limit >= 0 else 0)) % MOD
    return (f0[zero][one] + f1[zero][one]) % MOD

print('sai :', wrong_base(1, 2, 1), '| đúng:', Solution().numberOfStableArrays(1, 2, 1))

sai : 2 | đúng: 1


**Vì sao sai:** mảng toàn `1` dài `j > limit` đã vi phạm; coi nó hợp lệ làm phép trừ phía sau dựa trên dữ liệu sai (`[1, 1, 0]` bị tính).

### Lỗi 3 — Quên phép trừ (đếm mọi mảng)

In [14]:
def wrong_no_subtract(zero, one, limit):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, min(zero, limit) + 1): f0[i][0] = 1
    for j in range(1, min(one, limit) + 1): f1[0][j] = 1
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            f0[i][j] = (f0[i - 1][j] + f1[i - 1][j]) % MOD                                              # SAI
            f1[i][j] = (f0[i][j - 1] + f1[i][j - 1]) % MOD
    return (f0[zero][one] + f1[zero][one]) % MOD

print('sai :', wrong_no_subtract(3, 3, 2), '| đúng:', Solution().numberOfStableArrays(3, 3, 2))

sai : 18 | đúng: 14


**Vì sao sai:** không có gì ngăn khối kéo dài quá `limit` ở giữa mảng (chỉ hàng cơ sở bị giới hạn).

### Lỗi 4 — Tổ hợp chỉ ghép `a = b` khối

In [15]:
def wrong_equal_blocks(zero, one, limit):
    return sum(2 * parts(zero, k, limit) * parts(one, k, limit) for k in range(1, min(zero, one) + 1)) % MOD   # SAI

print('sai :', wrong_equal_blocks(1, 2, 1), '| đúng:', Solution().numberOfStableArrays(1, 2, 1))

sai : 0 | đúng: 1


**Vì sao sai:** mảng có thể bắt đầu và kết thúc bằng cùng một chữ số (`1, 0, 1`: hai khối 1, một khối 0).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi đoạn dài hơn `L` chứa cả hai loại" ⇔ "không có khối dài hơn `L`".
- DP thêm khối + tổng trượt ⇒ thay bằng phép trừ một số hạng.
- Chia `n` thành `k` phần bị chặn trên ⇒ inclusion–exclusion với `C(n − tL − 1, k − 1)`.

## 15. Biến thể của bài

1. **LC 3129 (bản I):** `≤ 200` — DP `O(z · o · limit)` đủ.
2. Giới hạn khác nhau cho 0 và 1 (`limit0`, `limit1`) ⇒ hai phép trừ dùng hai giới hạn.
3. Bảng chữ cái `k` ký tự, mỗi ký tự có số lượng cố định ⇒ DP đa chiều (bùng nổ trạng thái).
4. Đếm mảng **tuần hoàn** (vòng) ổn định ⇒ cần xử lý khối nối đầu–cuối.

In [16]:
# Biến thể 2: giới hạn riêng cho khối 0 và khối 1
def two_limits(zero, one, L0, L1):
    f0 = [[0] * (one + 1) for _ in range(zero + 1)]; f1 = [[0] * (one + 1) for _ in range(zero + 1)]
    for i in range(1, min(zero, L0) + 1): f0[i][0] = 1
    for j in range(1, min(one, L1) + 1): f1[0][j] = 1
    for i in range(1, zero + 1):
        for j in range(1, one + 1):
            f0[i][j] = (f0[i - 1][j] + f1[i - 1][j] - (f1[i - 1 - L0][j] if i - 1 - L0 >= 0 else 0)) % MOD
            f1[i][j] = (f0[i][j - 1] + f1[i][j - 1] - (f0[i][j - 1 - L1] if j - 1 - L1 >= 0 else 0)) % MOD
    return (f0[zero][one] + f1[zero][one]) % MOD

def two_limits_brute(zero, one, L0, L1):
    n = zero + one; cnt = 0
    for pos in itertools.combinations(range(n), zero):
        arr = [1] * n
        for p in pos: arr[p] = 0
        cnt += all(len(list(g)) <= (L0 if k == 0 else L1) for k, g in itertools.groupby(arr))
    return cnt

for _ in range(200):
    z, o = rng.randint(1, 7), rng.randint(1, 7); L0, L1 = rng.randint(1, 4), rng.randint(1, 4)
    assert two_limits(z, o, L0, L1) == two_limits_brute(z, o, L0, L1)
print('Biến thể hai giới hạn: khớp brute.')

Biến thể hai giới hạn: khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê | mũ | Oracle |
| 2 | DP thêm khối | `O(z·o·L)` | Bản I |
| 3 | DP trừ | `O(z·o)` | **Nộp bài** |
| 4 | Tổ hợp + inclusion–exclusion | `≈ O(z² + o²)` | Không cần bảng 2 chiều |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `(2, 2, 2)`.

**Bài 2 (tính tay):** `(2, 3, 1)`.

**Bài 3 (điền TODO):** liệt kê mọi mảng ổn định theo thứ tự từ điển cho tham số nhỏ.

**Bài 4 (tự viết lại):** DP trừ với **bộ nhớ `O(one)`** (chỉ giữ `limit + 1` hàng gần nhất).

**Bài 5 (mini challenge):** đếm mảng **vòng** ổn định (coi đầu–cuối kề nhau; xoay khác nhau tính là khác).

In [17]:
def list_stable_todo(zero, one, limit):
    # TODO
    raise NotImplementedError

def low_memory_todo(zero, one, limit):
    # TODO
    raise NotImplementedError

def circular_todo(zero, one, limit):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `limit = 2 =` số lượng mỗi loại. (2) Mọi `C(4, 2)` mảng. (3) `6`.

**Bài 2.** (1) `limit = 1` ⇒ xen kẽ hoàn toàn. (2) Ba số 1, hai số 0: `1 0 1 0 1`. (3) `1`.

**Bài 3.** (1) Quay lui với số 0/1 còn lại và độ dài khối cuối. (2) Ưu tiên 0 trước 1. (3) Số lượng phải bằng đáp án DP.

**Bài 4.** (1) `f0[i]` chỉ cần các hàng `i − 1` và `i − 1 − limit` ⇒ giữ hàng đợi `limit + 1` hàng. (2) `collections.deque(maxlen = limit + 2)`. (3) So với bản đầy đủ.

**Bài 5.** (1) Trên vòng, khối đầu và cuối cùng chữ số thì nối lại. (2) Brute: kiểm mảng nhân đôi. (3) Với tham số nhỏ, đếm bằng brute là đủ cho bài tập này.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().numberOfStableArrays(2, 2, 2) == 6
assert Solution().numberOfStableArrays(2, 3, 1) == 1

def list_stable(zero, one, limit):
    out = []
    def go(z, o, last, run, cur):
        if z == 0 and o == 0:
            out.append(cur[:]); return
        for d in (0, 1):
            if (d == 0 and z == 0) or (d == 1 and o == 0):
                continue
            nr = run + 1 if d == last else 1
            if nr > limit:
                continue
            cur.append(d); go(z - (d == 0), o - (d == 1), d, nr, cur); cur.pop()
    go(zero, one, -1, 0, [])
    return out

for _ in range(100):
    z, o, L = rng.randint(1, 6), rng.randint(1, 6), rng.randint(1, 4)
    lst = list_stable(z, o, L)
    assert lst == sorted(lst) and len(lst) == Solution().numberOfStableArrays(z, o, L)
print('Bài 1: 6 | Bài 2: 1 | Bài 3: (2, 2, 1) ->', list_stable(2, 2, 1))

Bài 1: 6 | Bài 2: 1 | Bài 3: (2, 2, 1) -> [[0, 1, 0, 1], [1, 0, 1, 0]]


In [19]:
# Bài 4, 5
from collections import deque
def low_memory(zero, one, limit):
    rows = deque(maxlen=limit + 2)                     # mỗi phần tử: (f0_row, f1_row) theo i
    r0 = [0] * (one + 1); r1 = [0] * (one + 1)
    for j in range(1, min(one, limit) + 1): r1[j] = 1
    rows.append((r0, r1))
    for i in range(1, zero + 1):
        p0, p1 = rows[-1]
        old = rows[-1 - limit] if len(rows) > limit else None    # hàng i - 1 - limit
        c0 = [0] * (one + 1); c1 = [0] * (one + 1)
        c0[0] = 1 if i <= limit else 0
        for j in range(1, one + 1):
            c0[j] = (p0[j] + p1[j] - (old[1][j] if old is not None else 0)) % MOD
            c1[j] = (c0[j - 1] + c1[j - 1] - (c0[j - 1 - limit] if j - 1 - limit >= 0 else 0)) % MOD
        rows.append((c0, c1))
    c0, c1 = rows[-1]
    return (c0[one] + c1[one]) % MOD

def circular(zero, one, limit):
    n = zero + one; cnt = 0
    for pos in itertools.combinations(range(n), zero):
        arr = [1] * n
        for p in pos: arr[p] = 0
        if all(arr[i] == arr[0] for i in range(n)):
            cnt += n <= limit; continue
        k = next(i for i in range(n) if arr[i] != arr[i - 1])      # xoay để bắt đầu tại ranh giới khối
        rot = arr[k:] + arr[:k]
        cnt += stable_by_runs(rot, limit)
    return cnt

for _ in range(150):
    z, o, L = rng.randint(1, 12), rng.randint(1, 12), rng.randint(1, 6)
    assert low_memory(z, o, L) == Solution().numberOfStableArrays(z, o, L)
assert circular(2, 2, 1) == 2 and circular(1, 1, 1) == 2
print('Bài 4 (bộ nhớ O(one)) khớp lời giải | Bài 5 (vòng, brute): (2,2,1) ->', circular(2, 2, 1), '| (3,3,2) ->', circular(3, 3, 2))

Bài 4 (bộ nhớ O(one)) khớp lời giải | Bài 5 (vòng, brute): (2,2,1) -> 2 | (3,3,2) -> 14


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f0[i][0] = 1 (1 ≤ i ≤ limit); f1[0][j] = 1 (1 ≤ j ≤ limit)
f0[i][j] = f0[i-1][j] + f1[i-1][j] − f1[i-1-limit][j]
f1[i][j] = f0[i][j-1] + f1[i][j-1] − f0[i][j-1-limit]
ans = f0[z][o] + f1[z][o]   (mod, cẩn thận số âm)
```

**Checklist:**

- [ ] Hàng cơ sở giới hạn bởi `limit`?
- [ ] Trừ ở chỉ số `i − 1 − limit` (không phải `i − limit`)?
- [ ] `% MOD` sau phép trừ (Python tự xử lý số âm đúng)?
- [ ] Cộng cả `f0` và `f1` ở ô cuối?